# 任务2：原ResNet基线与优化实验

前半部分保留已经运行的原模型及临床融合，后半部分新增独立的优化版。已有基线结果直接保留；运行优化实验只需执行定义单元和最后的优化训练单元，不必再开原模型训练开关。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。

分类评价另外使用 ROC、混淆矩阵和重采样函数，将后续模型的概率输出整理成可比较的指标。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


In [ ]:
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


## 训练工具与基本设置

PyTorch 负责网络计算和训练，torchvision 提供带 ImageNet 预训练权重的 ResNet50，以及图像增强所需的仿射变换。`StratifiedKFold` 用于在划分时尽量保持各折的结局比例。

这里设定随机种子为 `20260922`，每批读取 16 位患者。`DEVICE` 根据环境选择 GPU 或 CPU，后续模型和输入张量都使用同一设备。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 双分支网络：MRI 特征与病灶特征怎样合并

每位患者输入 4 层影像。每一层把 T1-FLAIR、T2、DWI 作为三个通道，一起送入共享的 ResNet50，得到 2048 维影像特征；同层的三类病灶掩模送入 `MaskEncoder`，经过三组卷积和全局池化，得到 256 维病灶特征。

`DualBranchModel` 将两种特征拼成每层 2304 维的向量。注意力模块为四层分别计算权重，经过 softmax 后加权求和，形成患者级特征，再由分类头输出一个分数。`predict_proba` 在预测时通过 sigmoid 将分数转换成 PSCI 阳性概率。

| 阶段 | 每位患者的数据形状 |
| --- | --- |
| MRI 输入 | `4 × 3 × 224 × 224` |
| 病灶掩模输入 | `4 × 3 × 224 × 224` |
| 每层特征拼接 | `4 × (2048 + 256)` |
| 四层加权汇总 | `2304` |
| 分类输出 | 一个阳性概率 |

同一单元中的 `augment_batch` 用于训练增强：同一患者的四层影像和掩模使用一致的旋转、平移和缩放，保持空间对应；亮度变化与随机噪声只加在影像上。`seed_everything` 设置各个随机数来源，便于复现实验。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: seed_everything
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: DualBranchModel
class DualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


# 直接复制：code/train_resnet50_manual_cv.py :: augment_batch
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


# 直接复制：code/train_resnet50_manual_cv.py :: predict_proba
@torch.no_grad()
def predict_proba(
    model: nn.Module,
    X: torch.Tensor,
    M: torch.Tensor,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size].to(DEVICE)
        mb = M[i : i + batch_size].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outs.append(torch.sigmoid(model(xb, mb)).float().cpu().numpy())
    return np.concatenate(outs)


## 全队列随机五折的训练函数

`train_full_fold` 完成一折训练，根据该折留出患者的 AUC 保存最佳权重；`train_full_cv_main` 则在全部缓存患者上组织五折训练，汇总折外预测和各折指标。

这组函数用于复现早期的全队列随机五折结果，调用入口放在本 Notebook 末尾。下面的时间验证研究使用另一组开发队列训练函数，模型输出分别保存在 `full_cv/` 和 `temporal/`。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: train_fold
def train_full_fold(
    Xtr: torch.Tensor,
    Mtr: torch.Tensor,
    ytr: torch.Tensor,
    Xva: torch.Tensor,
    Mva: torch.Tensor,
    yva: torch.Tensor,
    fold: int,
) -> Model:
    model = DualBranchModel().to(DEVICE)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.image_features.parameters(), "lr": 1e-4},
            {
                "params": list(model.mask_features.parameters())
                + list(model.attention.parameters())
                + list(model.head.parameters()),
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )
    n_pos = int((ytr > 0.5).sum())
    n_neg = len(ytr) - n_pos
    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 0.5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_state, patience = 0.0, None, 0

    for epoch in range(1, 31):
        t0 = time.time()
        model.train()
        perm = torch.randperm(len(Xtr))
        epoch_loss = 0.0
        for i in range(0, len(Xtr), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = Xtr[bi], Mtr[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = ytr[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)
        epoch_loss /= len(Xtr)

        val_pred = predict_proba(model, Xva, Mva)
        val_auc = float(roc_auc_score(yva.numpy(), val_pred))
        if val_auc > best_auc:
            best_auc, best_state, patience = val_auc, copy.deepcopy(model.state_dict()), 0
        else:
            patience += 1
        if epoch % 3 == 0 or patience >= 8:
            print(
                f"fold {fold} epoch {epoch:02d} loss={epoch_loss:.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f} time={time.time()-t0:.1f}s",
                flush=True,
            )
        if patience >= 8:
            break

    model.load_state_dict(best_state)
    return model


# 直接复制：code/train_resnet50_manual_cv.py :: main
def train_full_cv_main() -> None:
    seed_everything(SEED)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    data = np.load(CACHE)
    images = torch.from_numpy(data["images"].astype(np.float32))
    masks = torch.from_numpy(data["masks"].astype(np.uint8))
    labels = data["labels"].astype(np.int64)
    mrs = [str(x) for x in data["mrs"]]

    print("data", images.shape, masks.shape, "pos", int(labels.sum()), "device", DEVICE, flush=True)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(labels), dtype=np.float32)
    fold_aucs = []
    fold_start = time.time()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
        ytr = torch.from_numpy(labels[tr_idx].astype(np.float32))
        yva = torch.from_numpy(labels[va_idx].astype(np.float32))
        model = train_full_fold(
            images[tr_idx],
            masks[tr_idx],
            ytr,
            images[va_idx],
            masks[va_idx],
            yva,
            fold,
        )
        pred = predict_proba(model, images[va_idx], masks[va_idx])
        oof[va_idx] = pred
        fold_auc = roc_auc_score(labels[va_idx], pred)
        fold_aucs.append(fold_auc)
        torch.save(model.state_dict(), OUT_DIR / f"fold_{fold}.pt")
        print(f"fold {fold} AUC={fold_auc:.4f}", flush=True)

    pooled_auc = roc_auc_score(labels, oof)
    fpr, tpr, thr = roc_curve(labels, oof)
    youden = thr[np.argmax(tpr - fpr)]
    pred = (oof >= youden).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    tn = int(((pred == 0) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    acc = (tp + tn) / len(labels)

    print("\n" + "=" * 60)
    print(f"fold AUCs={[round(a, 4) for a in fold_aucs]}")
    print(f"mean AUC={np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}")
    print(f"pooled AUC={pooled_auc:.4f} acc={acc:.4f} sens={sens:.4f} spec={spec:.4f} youden={youden:.4f}")
    print(f"elapsed={time.time()-fold_start:.1f}s")
    print("=" * 60)

    pd.DataFrame(
        {
            "mr": mrs,
            "label": labels,
            "resnet50_prob": oof,
        }
    ).to_csv(OUT_DIR / "resnet50_manual_oof.csv", index=False)
    print("saved OOF to", OUT_DIR / "resnet50_manual_oof.csv")


## 开发队列五折训练与最终模型重训

`make_optimizer` 对两个部分使用不同学习率：预训练影像分支为 `1e-4`，病灶分支、注意力和分类头为 `5e-4`，优化器为 AdamW。`make_criterion` 根据训练数据中的阴阳性比例设置加权二元交叉熵。

`train_development_fold` 每轮完成增强、前向计算、反向传播和留出折预测，最多训练 30 轮；连续 8 轮 AUC 未改善时停止，并恢复最佳权重。GPU 训练使用混合精度，同时限制梯度范数，控制显存占用和梯度幅度。

`train_development_main` 完成五折后，记录每位患者的折外预测（OOF）和各折最佳轮数，再取最佳轮数的中位数作为最终训练轮数。`train_final` 重新初始化网络，用全部开发患者训练一个最终模型，供时间验证使用。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_temporal_development.py :: make_optimizer
def make_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
        [
            {"params": model.image_features.parameters(), "lr": 1e-4},
            {
                "params": list(model.mask_features.parameters())
                + list(model.attention.parameters())
                + list(model.head.parameters()),
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )


# 直接复制：code/train_resnet50_temporal_development.py :: make_criterion
def make_criterion(y: np.ndarray | torch.Tensor) -> nn.BCEWithLogitsLoss:
    y_np = np.asarray(y)
    n_pos = int((y_np > 0.5).sum())
    n_neg = len(y_np) - n_pos
    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 0.5)], device=DEVICE)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)


# 直接复制：code/train_resnet50_temporal_development.py :: train_fold
def train_development_fold(
    Xtr: torch.Tensor,
    Mtr: torch.Tensor,
    ytr: torch.Tensor,
    Xva: torch.Tensor,
    Mva: torch.Tensor,
    yva: torch.Tensor,
    fold: int,
) -> tuple[DualBranchModel, int]:
    model = DualBranchModel().to(DEVICE)
    optimizer = make_optimizer(model)
    criterion = make_criterion(ytr.numpy())
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_state, best_epoch, patience = 0.0, None, 1, 0

    for epoch in range(1, 31):
        model.train()
        perm = torch.randperm(len(Xtr))
        epoch_loss = 0.0
        for i in range(0, len(Xtr), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = Xtr[bi], Mtr[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = ytr[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)

        val_pred = predict_proba(model, Xva, Mva)
        val_auc = float(roc_auc_score(yva.numpy(), val_pred))
        if val_auc > best_auc:
            best_auc, best_state, best_epoch, patience = val_auc, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            patience += 1
        if epoch % 3 == 0 or patience >= 8:
            print(
                f"fold {fold} epoch {epoch:02d} loss={epoch_loss/len(Xtr):.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f}",
                flush=True,
            )
        if patience >= 8:
            break

    model.load_state_dict(best_state)
    return model, best_epoch


# 直接复制：code/train_resnet50_temporal_development.py :: train_final
def train_final(
    X: torch.Tensor,
    M: torch.Tensor,
    y: torch.Tensor,
    epochs: int,
) -> DualBranchModel:
    seed_everything(SEED + 100)
    model = DualBranchModel().to(DEVICE)
    optimizer = make_optimizer(model)
    criterion = make_criterion(y.numpy())
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(X))
        epoch_loss = 0.0
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = X[bi], M[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = y[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)
        if epoch % 3 == 0 or epoch == epochs:
            print(f"final epoch {epoch:02d}/{epochs:02d} loss={epoch_loss/len(X):.4f}", flush=True)

    return model


# 直接复制：code/train_resnet50_temporal_development.py :: main
def train_development_main() -> None:
    TEMPORAL_OUT.mkdir(parents=True, exist_ok=True)
    cache = np.load(CACHE, allow_pickle=True)
    cache_mrs = np.asarray([str(x).upper() for x in cache["mrs"]])
    cache_index = {mr: i for i, mr in enumerate(cache_mrs)}
    split = pd.read_csv(TEMPORAL_SPLIT)
    split["mr"] = split["mr"].astype(str).str.upper()
    dev = split[split["split"] == "development"].reset_index(drop=True)
    idx = np.asarray([cache_index[mr] for mr in dev["mr"]], dtype=int)

    X = torch.from_numpy(cache["images"][idx].astype(np.float32))
    M = torch.from_numpy(cache["masks"][idx].astype(np.uint8))
    y = cache["labels"][idx].astype(np.int64)
    mrs = dev["mr"].tolist()
    print("development", X.shape, "pos", int(y.sum()), flush=True)

    seed_everything(SEED)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y), dtype=np.float32)
    fold_aucs = []
    best_epochs = []
    start = time.time()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), y), 1):
        ytr = torch.from_numpy(y[tr_idx].astype(np.float32))
        yva = torch.from_numpy(y[va_idx].astype(np.float32))
        model, best_epoch = train_development_fold(
            X[tr_idx], M[tr_idx], ytr, X[va_idx], M[va_idx], yva, fold
        )
        pred = predict_proba(model, X[va_idx], M[va_idx])
        oof[va_idx] = pred
        fold_aucs.append(roc_auc_score(y[va_idx], pred))
        best_epochs.append(best_epoch)
        torch.save(model.state_dict(), TEMPORAL_OUT / f"dev_fold_{fold}.pt")
        print(f"development fold {fold} AUC={fold_aucs[-1]:.4f} best_epoch={best_epoch}", flush=True)

    fpr, tpr, thr = roc_curve(y, oof)
    youden = float(thr[np.argmax(tpr - fpr)])
    pooled_auc = roc_auc_score(y, oof)
    pred = (oof >= youden).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    metrics = {
        "n": len(y),
        "n_positive": int(y.sum()),
        "fold_aucs": [round(float(a), 4) for a in fold_aucs],
        "mean_auc": float(np.mean(fold_aucs)),
        "std_auc": float(np.std(fold_aucs)),
        "pooled_auc": float(pooled_auc),
        "youden_threshold": youden,
        "sensitivity": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "accuracy": float((tp + tn) / len(y)),
        "best_epochs": [int(x) for x in best_epochs],
    }
    with open(TEMPORAL_OUT / "development_cv_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    pd.DataFrame({"mr": mrs, "label": y, "dev_oof_prob": oof}).to_csv(
        TEMPORAL_OUT / "development_oof.csv", index=False
    )
    print("development CV", metrics, flush=True)

    final_epochs = max(1, int(np.median(best_epochs)))
    final_model = train_final(X, M, torch.from_numpy(y.astype(np.float32)), final_epochs)
    torch.save(final_model.state_dict(), TEMPORAL_OUT / "final_development_model.pt")
    print("saved final model, epochs", final_epochs, "elapsed", time.time() - start, flush=True)


## 临床建模工具

临床模型使用逻辑回归。`SimpleImputer` 填补缺失值，`StandardScaler` 统一变量尺度，`Pipeline` 将这些步骤和分类器组合起来，使每次拟合都按相同顺序处理数据。


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 临床模型与 ResNet 联合临床模型

临床模型使用年龄、受教育年限、NIHSS、同型半胱氨酸和尿酸五个变量。`make_pipe` 将中位数填补、标准化和逻辑回归串起来，`fit_fold_with_inner_cv` 在内层五折中从六个候选 C 值中选择正则化参数。

融合模型在这五个变量前增加一列 ResNet 预测概率，再拟合逻辑回归，共六个输入。开发阶段使用 ResNet 的折外概率作为这一列，时间验证时使用最终 ResNet 模型产生的概率。

`evaluate` 负责外层五折评估并返回折外预测；`direct_evaluate` 根据一组已有概率计算 AUC 和阈值相关指标。


In [ ]:
from __future__ import annotations


# 直接复制：code/fusion_resnet50_manual_clinical5.py :: make_pipe
def make_pipe(C: float) -> Pipeline:
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=3000, solver="liblinear")),
        ]
    )


# 直接复制：code/fusion_resnet50_manual_clinical5.py :: fit_fold_with_inner_cv
def fit_fold_with_inner_cv(Xtr: np.ndarray, ytr: np.ndarray) -> Pipeline:
    best_auc, best_C = -1.0, 1.0
    for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
        inner = StratifiedKFold(5, shuffle=True, random_state=SEED)
        aucs = []
        for itr, iva in inner.split(Xtr, ytr):
            pipe = make_pipe(C)
            pipe.fit(Xtr[itr], ytr[itr])
            proba = pipe.predict_proba(Xtr[iva])[:, 1]
            if len(np.unique(ytr[iva])) > 1:
                aucs.append(roc_auc_score(ytr[iva], proba))
        if aucs and np.mean(aucs) > best_auc:
            best_auc, best_C = np.mean(aucs), C
    pipe = make_pipe(best_C)
    pipe.fit(Xtr, ytr)
    return pipe


# 直接复制：code/fusion_resnet50_manual_clinical5.py :: evaluate
def evaluate(name: str, X: np.ndarray, y: np.ndarray, skf: StratifiedKFold) -> pd.Series:
    oof = np.zeros(len(y), dtype=np.float32)
    fold_aucs = []
    for tr, va in skf.split(np.zeros(len(y)), y):
        pipe = fit_fold_with_inner_cv(X[tr], y[tr])
        oof[va] = pipe.predict_proba(X[va])[:, 1]
        fold_aucs.append(roc_auc_score(y[va], oof[va]))

    pooled = roc_auc_score(y, oof)
    fpr, tpr, thr = roc_curve(y, oof)
    youden = thr[np.argmax(tpr - fpr)]
    pred = (oof >= youden).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    result = pd.Series(
        {
            "model": name,
            "fold_aucs": [round(a, 4) for a in fold_aucs],
            "mean_auc": float(np.mean(fold_aucs)),
            "std_auc": float(np.std(fold_aucs)),
            "pooled_auc": float(pooled),
            "accuracy": float((tp + tn) / len(y)),
            "sensitivity": float(sens),
            "specificity": float(spec),
            "youden": float(youden),
        }
    )
    print(result.to_string())
    return result, oof


# 直接复制：code/fusion_resnet50_manual_clinical5.py :: direct_evaluate
def direct_evaluate(name: str, prob: np.ndarray, y: np.ndarray) -> pd.Series:
    pooled = roc_auc_score(y, prob)
    fpr, tpr, thr = roc_curve(y, prob)
    youden = thr[np.argmax(tpr - fpr)]
    pred = (prob >= youden).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    return pd.Series(
        {
            "model": name,
            "fold_aucs": np.nan,
            "mean_auc": np.nan,
            "std_auc": np.nan,
            "pooled_auc": float(pooled),
            "accuracy": float((tp + tn) / len(y)),
            "sensitivity": float(sens),
            "specificity": float(spec),
            "youden": float(youden),
        }
    )


## 确定分类阈值并整理指标

`threshold_from_development` 从开发预测中选择 Youden 指数最高的分类阈值。`classification_metrics` 根据固定阈值统计正确与错误分类数，计算准确率、敏感度、特异度、AUC 和 Brier 分数，并可通过重采样估计 AUC 区间。

这样可以同时观察模型区分两类患者的能力，以及采用当前阈值时的实际分类表现。


In [ ]:
from __future__ import annotations

def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])







def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result







## 把训练、融合和模型保存串起来

下面四个函数负责组织执行步骤：`configure` 设置文件路径；`run_resnet_training` 选择时间开发训练或全队列随机五折；`fit_original_clinical_and_fusion` 读取影像折外概率，训练并保存临床和融合模型；`lock_original_models` 保存模型、阈值、队列与缓存的文件校验信息。

这些函数定义后，由下面各步骤的开关分别调用。这样可以先完成影像训练，再继续临床融合，或直接读取已有结果。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p







def run_resnet_training(cfg, scope="development"):
    p = configure(cfg)
    if scope == "development":
        fresh(p["temporal"] / "development_oof.csv", p["temporal"] / "final_development_model.pt")
        train_development_main()
    elif scope == "all_legacy":
        fresh(p["full_cv"])
        train_full_cv_main()
    else:
        raise ValueError("scope应为development或all_legacy。")
    record(cfg, "resnet_" + scope, {"architecture": "original DualBranchModel", "calibration": "none",
           "validation_note": "保留原验证折选最佳epoch的方法；未声称全流程嵌套无偏。"})







def fit_original_clinical_and_fusion(cfg):
    """Move the development-only fitting from the original evaluator into notebook02."""
    import joblib
    from sklearn.model_selection import StratifiedKFold
    p, dev = paths(cfg), development(cfg)
    fresh(p["temporal"] / "clinical_fusion.joblib", p["temporal"] / "clinical_fusion_meta.json")
    oof = pd.read_csv(p["temporal"] / "development_oof.csv", dtype={"mr": str})
    if oof.mr.tolist() != dev.mr.tolist() or not np.array_equal(oof.label, dev.label):
        raise ValueError("影像OOF与开发队列患者顺序或标签不一致。")
    y = dev.label.to_numpy(dtype=int)
    clinical = dev[CLINICAL].to_numpy(dtype=np.float32)
    fused = np.column_stack([oof.dev_oof_prob.to_numpy(dtype=float), clinical])
    models, metadata, outputs = {}, {}, dev[["mr", "label", "development_fold"]].copy()
    for name, X in (("Clinical5", clinical), ("ResNet50_Clinical5", fused)):
        cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
        result, prob = evaluate(name, X, y, cv)
        models[name] = fit_fold_with_inner_cv(X, y)
        threshold = threshold_from_development(y, prob)
        if not np.isfinite(threshold):
            raise ValueError(f"{name}没有有限Youden阈值；请检查开发结果。")
        metadata[name] = {"threshold": threshold, "mean_fold_auc": float(result["mean_auc"]),
                          "pooled_auc": float(result["pooled_auc"]), "features": CLINICAL,
                          "protocol": "original_one_pass_oof_fusion_not_fully_nested"}
        outputs[name + "_prob"] = prob
    joblib.dump(models, p["temporal"] / "clinical_fusion.joblib")
    dump_json(p["temporal"] / "clinical_fusion_meta.json", metadata)
    outputs.to_csv(p["temporal"] / "clinical_fusion_development_oof.csv", index=False)
    record(cfg, "clinical_fusion", {"note": "保留原逻辑回归及一次OOF融合，不新增校准或融合方法。"})
    return metadata







def lock_original_models(cfg):
    p = paths(cfg)
    files = [p["temporal"] / name for name in ("final_development_model.pt", "development_cv_metrics.json",
             "clinical_fusion.joblib", "clinical_fusion_meta.json")]
    files += [p["cohort"], p["temporal"] / "temporal_split.csv"]
    lock = p["temporal"] / "frozen_artifacts.json"
    fresh(lock)
    mapping = {f.relative_to(p["run"]).as_posix(): digest(f) for f in files}
    dump_json(lock, {"artifacts": mapping, "notebook_definitions_sha256": source_snapshot(cfg), "calibration": "none",
                     "cache_sha256": digest(p["cache"]), "baseline_contract": baseline_contract(cfg)})
    return lock







In [ ]:
from __future__ import annotations

def baseline_contract(cfg):
    return definition_contract(cfg, "02", list(ORIGINAL_INFERENCE_CONTRACT))






ORIGINAL_INFERENCE_CONTRACT = {'MaskEncoder': 'b0a35f2673a19f24a5f492a757a65894b572a9819996d2aabb9572cc89b2de04', 'DualBranchModel': '75a158f454de456d8b6e452309c35eb101124771da83567a475d8af2e386bc95', 'predict_proba': '94646b415f42d60d4a16416a05f26afed9656dd0fba15b09323c50f226898663'}


## 1. 查看开发队列的五折分布

按折号汇总患者人数和阳性人数，确认每折都有两类患者，且规模大致接近。这里读取的是 00 已保存的队列，后续训练和消融都以此为参照。


In [ ]:
if p["cohort"].exists():
    dev = development(cfg)
    display(dev.groupby("development_fold").agg(n=("mr", "size"), positive=("label", "sum")))
print("模型与训练实现：直接位于本Notebook上方的代码单元")


## 2. 训练 ResNet

在训练服务器上启用此步骤后，会先训练五个开发折模型，再用全部开发患者训练最终模型。主要输出包括 `dev_fold_*.pt`、`development_oof.csv`、开发指标文件和 `final_development_model.pt`。

观察训练日志中的损失、留出折 AUC 和最佳轮数，判断训练是否收敛、各折差异是否过大。首次构建模型时需要能够读取 ResNet50 的 ImageNet 预训练权重。


In [ ]:
RUN_RESNET_TRAINING = False

if RUN_RESNET_TRAINING:
    run_resnet_training(cfg, scope="development")


## 3. 训练临床模型与融合模型

这一步读取上一阶段生成的 ResNet 折外概率，与五个临床变量按患者一一匹配，然后分别训练临床模型和融合模型。

输出包含两组开发指标、各自的分类阈值，以及 `clinical_fusion.joblib` 中保存的完整处理流程。后面验证时可直接使用其中的填补值、标准化参数和回归系数。


In [ ]:
RUN_CLINICAL_FUSION = False

if RUN_CLINICAL_FUSION:
    metadata = fit_original_clinical_and_fusion(cfg)
    display(metadata)


## 4. 查看开发结果

`mean_auc` 或 `mean_fold_auc` 表示五个折 AUC 的平均值，`pooled_auc` 表示把所有开发患者的折外概率合在一起计算的 AUC。两者的计算方式不同，比较模型时应使用相同口径。

同时查看各折结果和最终训练轮数，有助于发现某一折明显偏低，或模型很早停止训练等情况。


In [ ]:
import json
import pandas as pd
for filename in ["development_cv_metrics.json", "clinical_fusion_meta.json"]:
    target = p["temporal"] / filename
    if target.exists():
        print(filename)
        display(json.loads(target.read_text(encoding="utf-8")))
if (p["temporal"] / "development_oof.csv").exists():
    display(pd.read_csv(p["temporal"] / "development_oof.csv").head())


## 5. 保存用于时间验证的模型版本

开发完成后，将最终 ResNet、临床及融合模型、阈值、患者清单和缓存的校验信息写入 `frozen_artifacts.json`。06 会据此确认加载的是同一次实验的文件。

校验还记录 Notebook 中的函数和模型定义；修改说明文字不会改变这部分记录。需要重新开展实验时，使用新的 `run_id` 保存结果。


In [ ]:
LOCK_ORIGINAL_MODELS = False

if LOCK_ORIGINAL_MODELS:
    print(lock_original_models(cfg))


## 可选：复现全 206 例的随机五折结果

这条支线用于与早期随机五折结果核对。它会将全部缓存患者用于交叉验证，输出单独保存到 `full_cv/`。

时间验证仍使用前面仅在开发队列训练的最终模型，因此完成本节并不是运行 06 的前提。


In [ ]:
RUN_ALL_PATIENTS_LEGACY_CV = False
if RUN_ALL_PATIENTS_LEGACY_CV:
    run_resnet_training(cfg, scope="all_legacy")


## 怎样解读这组基线结果

开发折的标签参与了最佳训练轮数的选择，因此其 AUC 可能偏乐观；融合模型使用一次生成的影像折外概率，也尚未对整套影像训练与融合过程进行完全嵌套的评估。报告时应结合 06 的时间验证结果判断泛化表现。

下面的优化版以本节模型作为基线，网络与训练设置的变化分别写明，结果保存到独立目录。


## 优化版改了什么

| 部分 | 原模型 | 本次修改与作用 |
|---|---|---|
| 图像特征 | 2048维 | 映射到256维，减少融合阶段的自由度 |
| 病灶分支 | 256维、BatchNorm | 64维、GroupNorm，减少对小批次统计的依赖 |
| 四层汇总 | 单路Tanh注意力 | Tanh与Sigmoid门控相乘后计算切片权重 |
| ResNet训练 | 整个骨干一起微调 | 前3轮仅训练新模块，此后只微调layer4；骨干BatchNorm统计和参数固定 |
| 骨干学习率 | 1e-4 | layer4使用1e-5，新模块仍为5e-4 |
| 训练轮数选择 | 在外折上早停并报告AUC | 只在外折训练患者内留出20%选轮数，再重置模型、用全部外折训练患者训练并评估外折 |
| 概率保存 | 原推理精度 | Sigmoid前转float32，减少半精度概率舍入 |

仍沿用三序列、四层、224×224缓存、原增强、加权BCE、AdamW、梯度裁剪和固定时间划分。默认批大小16，最多30轮，耐心值8。最终模型使用五折选中轮数的中位数训练全部开发患者。

这些改动是一组待验证的候选方案，不预设AUC一定提高。由于轮数选择方式也改变，开发OOF的差异不能全部归因于结构；原03消融继续回答原模型各分支的作用。此版先优化影像＋病灶模型，原临床融合结果单独保留。门控注意力参考 [Ilse等，ICML 2018](https://arxiv.org/abs/1802.04712)。


In [ ]:
"""Small-cohort ResNet candidate; isolated outputs and inner epoch selection."""
from __future__ import annotations
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights


class GatedSliceAttention(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, features):
        return self.score(torch.tanh(self.value(features)) * torch.sigmoid(self.gate(features)))






class CompactMaskEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(.3))

    def forward(self, masks):
        return self.features(masks.float())






class OptimizedResNet(nn.Module):
    def __init__(self, settings, pretrained=True):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_projection = nn.Sequential(nn.Linear(2048, settings["image_dim"]),
            nn.LayerNorm(settings["image_dim"]), nn.GELU(), nn.Dropout(.3))
        self.mask_features = CompactMaskEncoder(settings["mask_dim"])
        features = settings["image_dim"] + settings["mask_dim"]
        self.attention = GatedSliceAttention(features, settings["attention_dim"])
        self.head = nn.Sequential(nn.LayerNorm(features), nn.Dropout(.4), nn.Linear(features, 1))

    def train(self, mode=True):
        super().train(mode)
        # Keep pretrained running means/variances stable with small patient batches.
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

    def set_training_stage(self, fine_tune):
        for parameter in self.image_features.parameters():
            parameter.requires_grad_(False)
        if fine_tune:
            for parameter in self.image_features[7].parameters():
                parameter.requires_grad_(True)
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                for parameter in module.parameters():
                    parameter.requires_grad_(False)

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image = self.image_features(images.reshape(b * slices, channels, h, w)).flatten(1)
        image = self.image_projection(image).reshape(b, slices, -1)
        mask = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
        features = torch.cat([image, mask], dim=-1)
        attention = torch.softmax(self.attention(features).squeeze(-1), dim=1)
        return self.head((features * attention.unsqueeze(-1)).sum(dim=1)).squeeze(-1)






def optimized_contract(cfg):
    return definition_contract(cfg, "02", ["GatedSliceAttention", "CompactMaskEncoder", "OptimizedResNet",
        "optimized_predict", "fit_optimized_epochs", "run_optimized_training", "optimized_data",
        "seed_everything", "augment_batch", "make_criterion"])






def optimized_data(cfg, scope):
    table = read_cohort(cfg).query("split == @scope").reset_index(drop=True)
    indices = table.cache_index.to_numpy(dtype=int)
    with np.load(paths(cfg)["cache"], allow_pickle=False) as cache:
        if cache["mrs"][indices].astype(str).tolist() != table.mr.tolist() or not np.array_equal(cache["labels"][indices], table.label):
            raise ValueError("缓存患者或结局与队列不一致。")
        if not np.array_equal(cache["max_layers"][indices], table.max_layer):
            raise ValueError("缓存取层与队列不一致。")
        images = torch.from_numpy(cache["images"][indices].astype(np.float32))
        masks = torch.from_numpy(cache["masks"][indices].astype(np.uint8))
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("优化版继续使用四层、三序列、224×224的原缓存。")
    return table, images, masks, table.label.to_numpy(dtype=int)






def optimized_predict(model, images, masks, batch_size=16):
    model.eval()
    result = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(images[start:start + batch_size].to(DEVICE), masks[start:start + batch_size].to(DEVICE))
            # Store float32 probabilities; avoid rounding at the final sigmoid in half precision.
            result.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(result)






def fit_optimized_epochs(settings, images, masks, y, train, epochs, seed, inner_valid=None):
    from sklearn.metrics import roc_auc_score
    seed_everything(seed)
    model = OptimizedResNet(settings).to(DEVICE)
    backbone = list(model.image_features.parameters())
    heads = [p for name, p in model.named_parameters() if not name.startswith("image_features.")]
    optimizer = torch.optim.AdamW([{"params": backbone, "lr": settings["backbone_lr"]},
        {"params": heads, "lr": settings["head_lr"]}], weight_decay=settings["weight_decay"])
    criterion = make_criterion(y[train])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_epoch, stale, history = -float("inf"), 1, 0, []
    for epoch in range(1, epochs + 1):
        model.set_training_stage(epoch > settings["warmup_epochs"])
        model.train()
        permutation = np.asarray(train)[torch.randperm(len(train)).numpy()]
        loss_sum = 0.0
        for start in range(0, len(train), settings["batch_size"]):
            indices = permutation[start:start + settings["batch_size"]]
            xb, mb = augment_batch(images[indices], masks[indices])
            target = torch.as_tensor(y[indices], dtype=torch.float32, device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb.to(DEVICE), mb.to(DEVICE)), target)
            if not torch.isfinite(loss):
                raise ValueError("训练损失出现非有限值，停止并检查输入。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach()) * len(indices)
        row = {"epoch": epoch, "loss": loss_sum / len(train)}
        if inner_valid is not None:
            probability = optimized_predict(model, images[inner_valid], masks[inner_valid], settings["batch_size"])
            auc = float(roc_auc_score(y[inner_valid], probability))
            row["inner_auc"] = auc
            if auc > best_auc:
                best_auc, best_epoch, stale = auc, epoch, 0
            else:
                stale += 1
        history.append(row)
        print(row, flush=True)
        if inner_valid is not None and stale >= settings["patience"] and epoch > settings["warmup_epochs"]:
            break
    return model, best_epoch if inner_valid is not None else epochs, history






def run_optimized_training(cfg):
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    p, settings = paths(cfg), dict(cfg["optimized"])
    if settings["batch_size"] < 1 or settings["max_epochs"] <= settings["warmup_epochs"] or not 0 < settings["inner_fraction"] < .5:
        raise ValueError("检查批大小、最大训练轮数、预热轮数和内部留出比例。")
    fresh(p["optimized"])
    table, images, masks, y = optimized_data(cfg, "development")
    folds = table.development_fold.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError("优化版沿用00固定的五折。")
    fingerprint = {"cohort_identity": cohort_identity(cfg), "cache_sha256": digest(p["cache"]), "code_contract": optimized_contract(cfg)}
    p["optimized"].mkdir(parents=True)
    dump_json(p["optimized"] / "training_config.json", {"settings": settings, "seed": SEED, **fingerprint,
        "protocol": "outer_patient_folds_inner_holdout_epoch_selection_refit", "clinical_fusion": "not_fitted"})
    oof, rows, selected_epochs = np.zeros(len(y), dtype=np.float32), [], []
    for fold in range(1, 6):
        outer_train, outer_valid = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        inner_train, inner_valid = train_test_split(outer_train, test_size=settings["inner_fraction"],
            stratify=y[outer_train], random_state=SEED + fold)
        selection = {"outer_train": table.mr.iloc[outer_train].tolist(), "outer_valid": table.mr.iloc[outer_valid].tolist(),
                     "inner_train": table.mr.iloc[inner_train].tolist(), "inner_valid": table.mr.iloc[inner_valid].tolist()}
        dump_json(p["optimized"] / f"fold_{fold}_patients.json", selection)
        probe, chosen, history = fit_optimized_epochs(settings, images, masks, y, inner_train,
            settings["max_epochs"], SEED + fold, inner_valid)
        del probe
        pd.DataFrame(history).to_csv(p["optimized"] / f"fold_{fold}_inner_history.csv", index=False)
        model, _, history = fit_optimized_epochs(settings, images, masks, y, outer_train, chosen, SEED + 100 + fold)
        oof[outer_valid] = optimized_predict(model, images[outer_valid], masks[outer_valid], settings["batch_size"])
        torch.save(model.state_dict(), p["optimized"] / f"fold_{fold}.pt")
        pd.DataFrame(history).to_csv(p["optimized"] / f"fold_{fold}_refit_history.csv", index=False)
        selected_epochs.append(chosen)
        rows.append({"fold": fold, "auc": float(roc_auc_score(y[outer_valid], oof[outer_valid])), "selected_epochs": chosen})
        del model
    threshold = threshold_from_development(y, oof)
    if not np.isfinite(threshold):
        raise ValueError("开发OOF没有形成有限阈值，请检查预测。")
    final_epochs = max(1, int(np.median(selected_epochs)))
    model, _, history = fit_optimized_epochs(settings, images, masks, y, np.arange(len(y)), final_epochs, SEED + 1000)
    torch.save(model.state_dict(), p["optimized"] / "final_model.pt")
    pd.DataFrame(history).to_csv(p["optimized"] / "final_history.csv", index=False)
    output = table[["mr", "label", "development_fold"]].copy()
    output["probability"] = oof
    output.to_csv(p["optimized"] / "development_oof.csv", index=False)
    pd.DataFrame(rows).to_csv(p["optimized"] / "fold_metrics.csv", index=False)
    aucs = [row["auc"] for row in rows]
    metrics = {**classification_metrics(y, oof, threshold, n_bootstrap=0), "pooled_auc": float(roc_auc_score(y, oof)),
        "mean_fold_auc": float(np.mean(aucs)), "std_fold_auc": float(np.std(aucs)), "fold_aucs": aucs,
        "selected_epochs": selected_epochs, "final_epochs": final_epochs}
    dump_json(p["optimized"] / "development_metrics.json", metrics)
    if fingerprint != {"cohort_identity": cohort_identity(cfg), "cache_sha256": digest(p["cache"]), "code_contract": optimized_contract(cfg)}:
        raise ValueError("训练期间输入或实现发生变化，不锁定模型。")
    artifacts = {file.name: digest(file) for file in p["optimized"].iterdir() if file.is_file()}
    dump_json(p["optimized"] / "frozen_artifacts.json", {**fingerprint, "artifacts": artifacts})
    record(cfg, "optimized_training", {"settings": settings, "metrics": metrics})
    return metrics






def verify_optimized(cfg):
    p = paths(cfg)
    lock = json.loads((p["optimized"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_identity"] != cohort_identity(cfg) or lock["cache_sha256"] != digest(p["cache"]) or lock["code_contract"] != optimized_contract(cfg):
        raise ValueError("优化模型与当前患者、缓存或实现不匹配。")
    for name, checksum in lock["artifacts"].items():
        if digest(p["optimized"] / name) != checksum:
            raise ValueError(f"优化模型文件发生变化：{name}")
    settings = json.loads((p["optimized"] / "training_config.json").read_text(encoding="utf-8"))["settings"]
    if settings != cfg["optimized"]:
        raise ValueError("优化参数与训练时不一致；加载已有模型时使用原参数，调参请更换 experiment。")
    return settings






def load_optimized_model(cfg, checkpoint=None):
    settings = verify_optimized(cfg)
    model = OptimizedResNet(settings, pretrained=False).to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["optimized"] / "final_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model






def predict_optimized_validation(cfg):
    table, images, masks, y = optimized_data(cfg, "validation")
    probability = optimized_predict(load_optimized_model(cfg), images, masks, cfg["optimized"]["batch_size"])
    meta = json.loads((paths(cfg)["optimized"] / "development_metrics.json").read_text(encoding="utf-8"))
    result = table[["mr", "label"]].copy()
    result["model"], result["probability"], result["threshold"] = "ResNet50_Optimized", probability, meta["threshold"]
    result["prediction"] = (probability >= meta["threshold"]).astype(int)
    result["protocol"] = "optimized_resnet_inner_epoch_selection"
    return result, pd.DataFrame([{"model": "ResNet50_Optimized", **classification_metrics(y, probability, meta["threshold"])}])






## 运行优化实验

参数在 `experiment_settings.json` 的 `optimized` 中。结果写到 `outputs/<run_id>/optimized/resnet_v1/`，包括内层选轮数记录、每折权重、OOF、最终模型和锁定清单。再次调参时更换 `optimized.experiment`，保留各次结果。外折预测只在该折训练结束后计算一次。


In [ ]:
RUN_OPTIMIZED_TRAINING = False
if RUN_OPTIMIZED_TRAINING:
    optimized_metrics = run_optimized_training(cfg)
    display(optimized_metrics)


## 查看原模型与优化版的开发指标

这里并列展示两者已保存的指标。时间验证和配对AUC比较在06完成，优化版的折外热图在04生成。


In [ ]:
for folder, filename in [(p["temporal"], "development_cv_metrics.json"), (p["optimized"], "development_metrics.json")]:
    file = folder / filename
    if file.exists():
        print(folder.name)
        display(json.loads(file.read_text(encoding="utf-8")))
